In [1]:
import gymnasium as gym
from time import sleep
import numpy as np

from tensorflow.keras import Sequential, layers, losses, metrics, optimizers, activations, callbacks
import matplotlib.pyplot as plt
from matplotlib import animation
from tensorflow import GradientTape
import tensorflow as tf

### Task 8: Using Gradient Policy train LunarLander-v2

## Task 9: Using TF-Agents play with SpaceInvanders-v4

## Task 10: Buy Raspberry PI3 + additional hardware
1. Install TF on Pi
2. See GoPiGo / BrickPi
3. Use robot to detect the lithest corner at room / nearest object
4. Detect humans in the room and move to

1. [Original Manual bt TensorFlow](https://www.tensorflow.org/install/pip)
2. [TF Benchmarking](https://www.hackster.io/news/benchmarking-tensorflow-and-tensorflow-lite-on-raspberry-pi-5-b9156d58a6a2)
3. [Useful Data ( books/hardware etc.) ](https://blog.arducam.com/raspberry-pi-pico-machine-learning/)

### Task 8: Using Gradient Policy train LunarLander-v2*

In [2]:
# Action Space
# There are four discrete actions available:
# 0: do nothing
# 1: fire left orientation engine
# 2: fire main engine
# 3: fire right orientation engine

# Observation Space
# The state space consists of 8 variables:
# 0: x-coordinate of the lander
# 1: y-coordinate of the lander
# 2: x-velocity of the lander
# 3: y-velocity of the lander
# 4: angle of the lander
# 5: angular velocity of the lander
# 6: whether the left leg has contact with the ground (1 if yes, else 0)
# 7: whether the right leg has contact with the ground (1 if yes, else 0)

In [3]:
TOTAL_EPISODES = 300

In [4]:
def print_observation(obs):
    feature_names = [
        "x_position",
        "y_position",
        "x_velocity",
        "y_velocity",
        "angle",
        "angular_velocity",
        "left_leg_contact",
        "right_leg_contact"
    ]

    for i, (name, value) in enumerate(zip(feature_names, obs)):
        print(f"obs[{i}] -> {name:20s}: {value}")

In [5]:
env = gym.make("LunarLander-v3", render_mode="human")
obs, info = env.reset(seed=42)
tota_reward = 0
for cur_step in range(TOTAL_EPISODES):
    # this is where you would insert your policy
    action = env.action_space.sample() # artificial random policy, replace with your own
    # receiving the next observation, reward and if the episode has terminated or truncated
    obs, reward, terminated, truncated, info = env.step(action)
    # optional: slow it a bit so you can see what's happening
    sleep(1/60)
    # If the episode has ended then we can reset to start a new episode
    if terminated or truncated:
        obs, info = env.reset()
    tota_reward += reward
env.close()
print(f'Total reward: {tota_reward}') # currently around -250, but should be around 200-300 with some tuning

Total reward: -640.5660023255966


## Manual politics

In [6]:
def do_action(obs):
    x_pos, y_pos, x_vel, y_vel, angle, angular_vel, left_leg_contact, right_leg_contact = obs
    # 0: do nothing
    # 1: fire left orientation engine
    # 2: fire main engine
    # 3: fire right orientation engine

    # Strong descent -> main engine
    if y_vel < -0.8:
        return 2

    # Near the ground, be more conservative
    if y_pos < 0.35 and y_vel < -0.25:
        return 2

    # If tilted, correct angle
    if angle > 0.12:
        return 3
    if angle < -0.12:
        return 1

    # If spinning, damp angular velocity
    if angular_vel > 0.25:
        return 3
    if angular_vel < -0.25:
        return 1

    # If drifting sideways, slightly steer against drift
    if x_vel > 0.3:
        return 3
    if x_vel < -0.3:
        return 1

    return 0


env = gym.make("LunarLander-v3", render_mode="human")
obs, info = env.reset(seed=42)
tota_reward = 0
for cur_step in range(TOTAL_EPISODES):
    # simple heuristic policy based on the angle of the lander

    action = do_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    sleep(1/60)
    if terminated or truncated:
        obs, info = env.reset()
    tota_reward += reward
env.close()
print(f'Total reward: {tota_reward}') # -0.44 currently, but should be around 200-300 with some tuning

Total reward: -44.23102792454655


## Gradient Policy for LunarLander-v2

>  REINFORCE ( REward Increment = non-negative Factor x Offset Reinforcement )

1. Play Env with model and calculate gradient ( but do not apply gradients )
2. After few runs calculate importance for each a actions
3. Increase or decrease probability of next action by good \ bad results
4. You can use calculated gradients

## REINFORCE / Policy Gradient
> Important difference from CartPole:
> - CartPole can be modeled with 2 actions and a sigmoid output
> - LunarLander has 4 actions, so we need 4 logits


In [7]:
ENV_NAME = "LunarLander-v3"
SEED = 42

In [8]:
env = gym.make(ENV_NAME, render_mode="rgb_array")
obs, info = env.reset(seed=SEED)

input_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

model = Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(n_actions, activation=),  # logits for 4 actions
])

# Build once so summary works immediately
_ = model(np.zeros((1, input_dim), dtype=np.float32))
model.summary()

env.close()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,180 (71.02 KB)

 Trainable params: 18,180 (71.02 KB)

 Non-trainable params: 0 (0.00 B)